# 18 v3. Public GIS domain features with rate-limit-safe elevation

This notebook creates external GIS/domain features for HPAI first-occurrence risk mapping.

Main output for notebook 14:

```text
/content/drive/MyDrive/avian_influenza_project/processed/domain_features/grid_environment_features.csv
```

Compared with v2, this version avoids excessive calls to the Open-Meteo Elevation API.

- Coast, river, lake/waterbody, and urban features are created from Natural Earth vectors.
- Elevation is obtained from cached SRTM data first.
- Open-Meteo Elevation API is disabled by default and used only as an optional slow fallback.
- Slope is calculated only when elevation is available. If not available, it is left as NaN.
- The notebook always saves partial valid features, even if elevation fails.


In [1]:
# ============================================================
# 18 v3. Public GIS source ingestion and real domain feature construction
# Version: 18_build_public_gis_domain_features_same_paths_v3_rate_limit_safe
# ============================================================

NOTEBOOK_VERSION = "18_build_public_gis_domain_features_same_paths_v3_rate_limit_safe"
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)

from pathlib import Path
import os, re, glob, json, math, zipfile, shutil, warnings, time
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Not running on Colab or Drive mount skipped:", e)

PROC_DIR = Path('/content/drive/MyDrive/avian_influenza_project/processed')
MODEL_DIR = PROC_DIR / 'model_outputs_riskmap_eval'
DOMAIN_DIR = PROC_DIR / 'domain_features'
GIS_RAW_DIR = PROC_DIR / 'gis_raw'
DOMAIN_RAW_DIR = PROC_DIR / 'domain_raw'
ENV_RAW_DIR = PROC_DIR / 'environment_raw'
DOWNLOAD_DIR = PROC_DIR / 'gis_downloads'
UNZIP_DIR = PROC_DIR / 'gis_unzipped'
CACHE_DIR = PROC_DIR / 'cache'
SRTM_CACHE_DIR = CACHE_DIR / 'srtm'

for d in [PROC_DIR, MODEL_DIR, DOMAIN_DIR, GIS_RAW_DIR, DOMAIN_RAW_DIR, ENV_RAW_DIR, DOWNLOAD_DIR, UNZIP_DIR, CACHE_DIR, SRTM_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROC_DIR:", PROC_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("DOMAIN_DIR:", DOMAIN_DIR)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
USE_NATURAL_EARTH_AUTO_DOWNLOAD = True
USE_SRTM_ELEVATION = True
USE_OPEN_METEO_FALLBACK = False  # Keep False unless you explicitly need it.
OPEN_METEO_CHUNK_SIZE = 25
OPEN_METEO_SLEEP_SEC = 2.0
OPEN_METEO_MAX_REQUESTS = 80

# Slope from SRTM is computed approximately by sampling four neighboring points.
# This does not call Open-Meteo. It only uses SRTM cache.
COMPUTE_SLOPE_FROM_SRTM = True
SLOPE_OFFSET_DEG = 0.01  # roughly 1 km latitude direction.


NOTEBOOK VERSION: 18_build_public_gis_domain_features_same_paths_v3_rate_limit_safe
Mounted at /content/drive
PROC_DIR: /content/drive/MyDrive/avian_influenza_project/processed
MODEL_DIR: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval
DOMAIN_DIR: /content/drive/MyDrive/avian_influenza_project/processed/domain_features


## 1. Install and import packages

In [2]:
import sys, subprocess, importlib.util

def ensure_package(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    else:
        print(f"{pkg} already available")

for pkg, imp in [
    ("geopandas", "geopandas"),
    ("pyogrio", "pyogrio"),
    ("rtree", "rtree"),
    ("scikit-learn", "sklearn"),
    ("requests", "requests"),
    ("srtm.py", "srtm"),
]:
    try:
        ensure_package(pkg, imp)
    except Exception as e:
        print("Package install/check failed:", pkg, e)

import requests
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree

try:
    import srtm
except Exception as e:
    srtm = None
    print("srtm not available:", e)

print("Packages ready.")


geopandas already available
pyogrio already available
rtree already available
scikit-learn already available
requests already available
Installing srtm.py ...
Packages ready.


## 2. Load grid master

In [3]:
candidate_grid_files = [
    MODEL_DIR / "15_grid_master_for_domain_features.csv",
    MODEL_DIR / "14_grid_master_for_domain_features.csv",
    MODEL_DIR / "17_grid_environment_features_for_14.csv",
    MODEL_DIR / "18_grid_environment_features_for_14.csv",
    DOMAIN_DIR / "grid_environment_features.csv",
]

grid_master = None
grid_source = None
for p in candidate_grid_files:
    if p.exists():
        try:
            tmp = pd.read_csv(p)
            if {"grid_id", "grid_lat", "grid_lon"}.issubset(tmp.columns):
                grid_master = tmp[["grid_id", "grid_lat", "grid_lon"]].drop_duplicates("grid_id").copy()
                grid_source = p
                break
        except Exception as e:
            print("Failed to read", p, e)

if grid_master is None:
    raise FileNotFoundError("Could not find grid master with grid_id, grid_lat, grid_lon.")

grid_master["grid_lat"] = pd.to_numeric(grid_master["grid_lat"], errors="coerce")
grid_master["grid_lon"] = pd.to_numeric(grid_master["grid_lon"], errors="coerce")
grid_master = grid_master.dropna(subset=["grid_lat", "grid_lon"]).reset_index(drop=True)

print("grid_source:", grid_source)
print("grid_master shape:", grid_master.shape)
grid_master.head()


grid_source: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/15_grid_master_for_domain_features.csv
grid_master shape: (5491, 3)


,grid_id,grid_lat,grid_lon
0,G000015,24.250542,123.786000
1,G000016,24.332420,123.786000
2,G000033,24.414246,124.235160
3,G000048,24.822575,125.313140
4,G000080,26.200762,127.738594


## 3. Utility functions

In [4]:
def make_grid_gdf(grid_df):
    return gpd.GeoDataFrame(
        grid_df.copy(),
        geometry=gpd.points_from_xy(grid_df["grid_lon"], grid_df["grid_lat"]),
        crs="EPSG:4326"
    )

def haversine_nearest_distance_km(src_lat, src_lon, dst_lat, dst_lon):
    src = np.deg2rad(np.c_[src_lat, src_lon])
    dst = np.deg2rad(np.c_[dst_lat, dst_lon])
    if len(dst) == 0:
        return np.full(len(src), np.nan)
    tree = BallTree(dst, metric="haversine")
    dist, ind = tree.query(src, k=1)
    return dist[:, 0] * 6371.0088

def count_within_radius(src_lat, src_lon, dst_lat, dst_lon, radius_km):
    src = np.deg2rad(np.c_[src_lat, src_lon])
    dst = np.deg2rad(np.c_[dst_lat, dst_lon])
    if len(dst) == 0:
        return np.zeros(len(src), dtype=int)
    tree = BallTree(dst, metric="haversine")
    ind = tree.query_radius(src, r=radius_km / 6371.0088)
    return np.array([len(x) for x in ind], dtype=int)

def distance_to_vector_km(grid_gdf, vec_gdf):
    if vec_gdf is None or vec_gdf.empty:
        return np.full(len(grid_gdf), np.nan)
    if vec_gdf.crs is None:
        vec_gdf = vec_gdf.set_crs("EPSG:4326", allow_override=True)
    vec_gdf = vec_gdf[vec_gdf.geometry.notna()].copy()
    vec_gdf = vec_gdf[~vec_gdf.geometry.is_empty].copy()
    if vec_gdf.empty:
        return np.full(len(grid_gdf), np.nan)
    grid_m = grid_gdf.to_crs("EPSG:3857")
    vec_m = vec_gdf.to_crs("EPSG:3857")
    geom_union = vec_m.geometry.union_all() if hasattr(vec_m.geometry, 'union_all') else vec_m.geometry.unary_union
    return grid_m.geometry.distance(geom_union).values / 1000.0

def ratio_within_vector_buffer(grid_gdf, vec_gdf, buffer_km=5.0):
    # Approximation: 1 if grid centroid is within 5km of vector/polygon, else 0.
    # For true area-weighted ratios, use land-use mesh/polygon data.
    if vec_gdf is None or vec_gdf.empty:
        return np.full(len(grid_gdf), np.nan)
    d = distance_to_vector_km(grid_gdf, vec_gdf)
    return (d <= buffer_km).astype(float)

FORBIDDEN_PATTERNS = [
    r"^y_lead", r"target", r"label", r"outbreak", r"pred", r"risk", r"rank",
    r"num_birds", r"same_grid_past", r"neighbor_outbreak", r"lag_outbreak", r"rolling_outbreak"
]

def is_forbidden_col(c):
    s = str(c).lower()
    return any(re.search(pat, s) for pat in FORBIDDEN_PATTERNS)

def detect_lat_lon_cols(df):
    cols = list(df.columns)
    lower = {c: str(c).lower() for c in cols}
    lat_candidates = [c for c in cols if lower[c] in ["lat", "latitude", "y", "緯度", "ido"] or "lat" in lower[c]]
    lon_candidates = [c for c in cols if lower[c] in ["lon", "lng", "longitude", "x", "経度", "keido"] or "lon" in lower[c] or "lng" in lower[c]]
    if lat_candidates and lon_candidates:
        return lat_candidates[0], lon_candidates[0]
    return None, None

grid_gdf = make_grid_gdf(grid_master)
domain_features = grid_master.copy()
creation_logs = []
print("Utilities ready.")


Utilities ready.


## 4. Auto-download Natural Earth vectors

This avoids manual GIS placement for the minimum viable external features.


In [5]:
NE_DIR = GIS_RAW_DIR / "natural_earth"
NE_DIR.mkdir(parents=True, exist_ok=True)

natural_earth_sources = [
    {
        "name": "coastline",
        "url": "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip",
        "zip_path": NE_DIR / "ne_10m_coastline.zip",
        "out_dir": NE_DIR / "ne_10m_coastline",
        "feature_column": "dist_to_coast_km",
    },
    {
        "name": "rivers_lake_centerlines",
        "url": "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip",
        "zip_path": NE_DIR / "ne_10m_rivers_lake_centerlines.zip",
        "out_dir": NE_DIR / "ne_10m_rivers_lake_centerlines",
        "feature_column": "dist_to_river_km",
    },
    {
        "name": "lakes",
        "url": "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_lakes.zip",
        "zip_path": NE_DIR / "ne_10m_lakes.zip",
        "out_dir": NE_DIR / "ne_10m_lakes",
        "feature_column": "dist_to_waterbody_km",
    },
    {
        "name": "urban_areas",
        "url": "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_urban_areas.zip",
        "zip_path": NE_DIR / "ne_10m_urban_areas.zip",
        "out_dir": NE_DIR / "ne_10m_urban_areas",
        "feature_column": "urban_ratio_5km",
    },
]

download_logs = []
if USE_NATURAL_EARTH_AUTO_DOWNLOAD:
    for src in natural_earth_sources:
        zpath = src["zip_path"]
        out_dir = src["out_dir"]
        try:
            if not zpath.exists():
                print("Downloading", src["name"])
                r = requests.get(src["url"], timeout=120)
                r.raise_for_status()
                zpath.write_bytes(r.content)
                status = "downloaded"
            else:
                status = "cached_zip"
            out_dir.mkdir(parents=True, exist_ok=True)
            if not list(out_dir.rglob("*.shp")):
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(out_dir)
            download_logs.append({"name": src["name"], "url": src["url"], "status": status, "zip_path": str(zpath), "out_dir": str(out_dir)})
        except Exception as e:
            download_logs.append({"name": src["name"], "url": src["url"], "status": "failed", "error": str(e)})
else:
    download_logs.append({"name": "natural_earth", "status": "disabled"})

pd.DataFrame(download_logs).to_csv(MODEL_DIR / "18_v3_natural_earth_download_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(download_logs)


,name,url,status,zip_path,out_dir
0,coastline,https://naturalearth.s3.amazonaws.com/10m_phys...,downloaded,/content/drive/MyDrive/avian_influenza_project...,/content/drive/MyDrive/avian_influenza_project...
1,rivers_lake_centerlines,https://naturalearth.s3.amazonaws.com/10m_phys...,downloaded,/content/drive/MyDrive/avian_influenza_project...,/content/drive/MyDrive/avian_influenza_project...
2,lakes,https://naturalearth.s3.amazonaws.com/10m_phys...,downloaded,/content/drive/MyDrive/avian_influenza_project...,/content/drive/MyDrive/avian_influenza_project...
3,urban_areas,https://naturalearth.s3.amazonaws.com/10m_cult...,downloaded,/content/drive/MyDrive/avian_influenza_project...,/content/drive/MyDrive/avian_influenza_project...


## 5. Create Natural Earth distance / proximity features

In [6]:
ne_feature_logs = []
for src in natural_earth_sources:
    out_dir = src["out_dir"]
    shp_files = list(out_dir.rglob("*.shp"))
    if not shp_files:
        ne_feature_logs.append({"source": src["name"], "status": "no_shp_found"})
        continue
    try:
        gdf = gpd.read_file(shp_files[0])
        # Rough Japan bbox with margin to reduce geometries.
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326", allow_override=True)
        gdf = gdf.to_crs("EPSG:4326")
        try:
            gdf = gdf.cx[120:155, 20:50]
        except Exception:
            pass
        if gdf.empty:
            ne_feature_logs.append({"source": src["name"], "status": "empty_after_bbox", "path": str(shp_files[0])})
            continue
        col = src["feature_column"]
        if col == "urban_ratio_5km":
            domain_features[col] = ratio_within_vector_buffer(grid_gdf, gdf, buffer_km=5.0)
            status = "created_proximity_ratio"
        else:
            domain_features[col] = distance_to_vector_km(grid_gdf, gdf)
            status = "created_distance"
            if col == "dist_to_waterbody_km":
                domain_features["dist_to_lake_or_reservoir_km"] = domain_features[col]
                domain_features["waterbody_ratio_5km"] = (domain_features[col] <= 5.0).astype(float)
        ne_feature_logs.append({"source": src["name"], "status": status, "column": col, "path": str(shp_files[0]), "n_features": len(gdf)})
    except Exception as e:
        ne_feature_logs.append({"source": src["name"], "status": "failed", "error": str(e)})

pd.DataFrame(ne_feature_logs).to_csv(MODEL_DIR / "18_v3_natural_earth_feature_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(ne_feature_logs)


,source,status,column,path,n_features
0,coastline,created_distance,dist_to_coast_km,/content/drive/MyDrive/avian_influenza_project...,241
1,rivers_lake_centerlines,created_distance,dist_to_river_km,/content/drive/MyDrive/avian_influenza_project...,36
2,lakes,created_distance,dist_to_waterbody_km,/content/drive/MyDrive/avian_influenza_project...,11
3,urban_areas,created_proximity_ratio,urban_ratio_5km,/content/drive/MyDrive/avian_influenza_project...,696


## 6. Optional raw file ingestion

If you later place better Japanese GIS/CSV files under `processed/gis_raw`, `processed/domain_raw`, or `processed/environment_raw`, this cell merges them without treating pipeline outputs as raw data.


In [7]:
generated_patterns = [
    r"/model_outputs_riskmap_eval/",
    r"/domain_features/grid_environment_features\.",
    r"/domain_features/15_", r"/domain_features/16_", r"/domain_features/17_", r"/domain_features/18_",
    r"/15_", r"/16_", r"/17_", r"/18_", r"/14_", r"/13_", r"/12_", r"/10_",
]

def is_generated_pipeline_file(path: Path) -> bool:
    s = "/" + str(path).replace("\\", "/")
    return any(re.search(pat, s) for pat in generated_patterns)

def infer_feature_kind(path: Path):
    name = path.name.lower()
    if any(k in name for k in ["coast", "coastline", "shore", "海岸"]): return "coast"
    if any(k in name for k in ["river", "stream", "河川"]): return "river"
    if any(k in name for k in ["wetland", "ramsar", "湿地"]): return "wetland"
    if any(k in name for k in ["waterbody", "water_body", "lake", "reservoir", "pond", "湖沼", "水域"]): return "waterbody"
    if any(k in name for k in ["landuse", "land_use", "l03", "土地利用"]): return "landuse"
    if any(k in name for k in ["dem", "elevation", "標高"]): return "dem"
    if any(k in name for k in ["poultry", "chicken", "farm", "鶏", "養鶏"]): return "poultry"
    if any(k in name for k in ["migratory", "wildbird", "bird", "野鳥", "渡り鳥"]): return "wildbird"
    return "unknown"

search_roots = [GIS_RAW_DIR, DOMAIN_RAW_DIR, ENV_RAW_DIR]
suffixes = [".csv", ".parquet", ".geojson", ".json", ".shp", ".gpkg"]
raw_files = []
for root in search_roots:
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in suffixes and not is_generated_pipeline_file(p):
            # skip natural earth already processed here to avoid duplicate long work
            if "natural_earth" in str(p):
                continue
            raw_files.append(p)

inventory = pd.DataFrame([{
    "path": str(p), "name": p.name, "suffix": p.suffix.lower(), "kind": infer_feature_kind(p), "size_mb": p.stat().st_size/1024/1024
} for p in raw_files]) if raw_files else pd.DataFrame(columns=["path","name","suffix","kind","size_mb"])
inventory.to_csv(MODEL_DIR / "18_v3_raw_file_inventory.csv", index=False, encoding="utf-8-sig")
print("Additional raw files found:", len(raw_files))
inventory.head(20)


Additional raw files found: 0


,path,name,suffix,kind,size_mb


In [8]:
raw_ingest_logs = []

# grid_id tables
for p in raw_files:
    if p.suffix.lower() not in [".csv", ".parquet"]:
        continue
    try:
        df = pd.read_csv(p) if p.suffix.lower() == ".csv" else pd.read_parquet(p)
    except Exception as e:
        raw_ingest_logs.append({"path": str(p), "status": "read_failed", "error": str(e)})
        continue
    if "grid_id" in df.columns:
        safe_cols = []
        for c in df.columns:
            if c == "grid_id" or is_forbidden_col(c):
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                safe_cols.append(c)
        if safe_cols:
            add = df[["grid_id"] + safe_cols].drop_duplicates("grid_id")
            rename_map = {c: f"{p.stem}_{c}" for c in safe_cols if c in domain_features.columns}
            add = add.rename(columns=rename_map)
            domain_features = domain_features.merge(add, on="grid_id", how="left")
            raw_ingest_logs.append({"path": str(p), "status": "merged_grid_id_table", "columns": ",".join([c for c in add.columns if c != "grid_id"])})
        continue
    lat_col, lon_col = detect_lat_lon_cols(df)
    if lat_col and lon_col:
        kind = infer_feature_kind(p)
        pts = df[[lat_col, lon_col]].copy()
        pts[lat_col] = pd.to_numeric(pts[lat_col], errors="coerce")
        pts[lon_col] = pd.to_numeric(pts[lon_col], errors="coerce")
        pts = pts.dropna()
        pts = pts[(pts[lat_col].between(20, 47)) & (pts[lon_col].between(122, 154))]
        if pts.empty:
            raw_ingest_logs.append({"path": str(p), "status": "no_valid_points"})
            continue
        if kind == "poultry":
            domain_features["dist_to_poultry_farm_km"] = haversine_nearest_distance_km(domain_features.grid_lat, domain_features.grid_lon, pts[lat_col], pts[lon_col])
            domain_features["poultry_farm_count_10km"] = count_within_radius(domain_features.grid_lat, domain_features.grid_lon, pts[lat_col], pts[lon_col], 10)
            raw_ingest_logs.append({"path": str(p), "status": "created_poultry_point_features"})
        elif kind == "wildbird":
            domain_features["dist_to_wildbird_surveillance_site_km"] = haversine_nearest_distance_km(domain_features.grid_lat, domain_features.grid_lon, pts[lat_col], pts[lon_col])
            domain_features["wildbird_site_count_30km"] = count_within_radius(domain_features.grid_lat, domain_features.grid_lon, pts[lat_col], pts[lon_col], 30)
            raw_ingest_logs.append({"path": str(p), "status": "created_wildbird_point_features"})
        elif kind == "wetland":
            domain_features["dist_to_wetland_km"] = haversine_nearest_distance_km(domain_features.grid_lat, domain_features.grid_lon, pts[lat_col], pts[lon_col])
            raw_ingest_logs.append({"path": str(p), "status": "created_wetland_point_features"})

# vector features
for p in raw_files:
    if p.suffix.lower() not in [".geojson", ".json", ".shp", ".gpkg"]:
        continue
    kind = infer_feature_kind(p)
    try:
        vec = gpd.read_file(p)
        if vec.crs is None:
            vec = vec.set_crs("EPSG:4326", allow_override=True)
        vec = vec.to_crs("EPSG:4326")
        try:
            vec = vec.cx[120:155, 20:50]
        except Exception:
            pass
        if vec.empty:
            raw_ingest_logs.append({"path": str(p), "status": "vector_empty_after_bbox"})
            continue
        if kind == "coast": col = "dist_to_coast_km"
        elif kind == "river": col = "dist_to_river_km"
        elif kind == "waterbody": col = "dist_to_waterbody_km"
        elif kind == "wetland": col = "dist_to_wetland_km"
        elif kind == "wildbird": col = "dist_to_wildbird_surveillance_site_km"
        else: col = f"dist_to_{p.stem}_km"
        domain_features[col] = distance_to_vector_km(grid_gdf, vec)
        raw_ingest_logs.append({"path": str(p), "status": "created_vector_distance", "column": col})
    except Exception as e:
        raw_ingest_logs.append({"path": str(p), "status": "vector_failed", "error": str(e)})

pd.DataFrame(raw_ingest_logs).to_csv(MODEL_DIR / "18_v3_raw_ingest_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(raw_ingest_logs).tail(20)


""


## 7. Elevation and slope with SRTM cache first

This avoids the Open-Meteo 429 rate-limit problem.

If SRTM fails, the notebook keeps going and leaves `elevation_m` / `slope_deg` as missing rather than spamming API requests.


In [9]:
elev_log = []

def get_srtm_data():
    if srtm is None:
        return None
    try:
        # srtm.py stores cache under its default cache. Set env var if supported by the package.
        os.environ["SRTM_CACHE_DIR"] = str(SRTM_CACHE_DIR)
        return srtm.get_data(local_cache_dir=str(SRTM_CACHE_DIR))
    except TypeError:
        try:
            return srtm.get_data()
        except Exception as e:
            elev_log.append({"source": "srtm", "status": "get_data_failed", "error": str(e)})
            return None
    except Exception as e:
        elev_log.append({"source": "srtm", "status": "get_data_failed", "error": str(e)})
        return None

def srtm_elevations_for_points(data, lats, lons):
    vals = []
    for lat, lon in zip(lats, lons):
        try:
            v = data.get_elevation(float(lat), float(lon))
            vals.append(np.nan if v is None else float(v))
        except Exception:
            vals.append(np.nan)
    return np.array(vals, dtype=float)

if USE_SRTM_ELEVATION:
    try:
        data = get_srtm_data()
        if data is not None:
            print("Getting SRTM elevation for", len(domain_features), "grid centroids...")
            elev = srtm_elevations_for_points(data, domain_features["grid_lat"].values, domain_features["grid_lon"].values)
            if np.isfinite(elev).sum() > 0:
                domain_features["elevation_m"] = elev
                elev_log.append({"source": "srtm", "status": "created_elevation", "non_null": int(np.isfinite(elev).sum())})
            else:
                elev_log.append({"source": "srtm", "status": "no_valid_elevation"})

            if COMPUTE_SLOPE_FROM_SRTM and "elevation_m" in domain_features.columns:
                lat = domain_features["grid_lat"].values
                lon = domain_features["grid_lon"].values
                e_n = srtm_elevations_for_points(data, lat + SLOPE_OFFSET_DEG, lon)
                e_s = srtm_elevations_for_points(data, lat - SLOPE_OFFSET_DEG, lon)
                e_e = srtm_elevations_for_points(data, lat, lon + SLOPE_OFFSET_DEG)
                e_w = srtm_elevations_for_points(data, lat, lon - SLOPE_OFFSET_DEG)
                # distances in meters for offsets
                dy = 2 * SLOPE_OFFSET_DEG * 111_320.0
                dx = 2 * SLOPE_OFFSET_DEG * 111_320.0 * np.cos(np.deg2rad(lat))
                dzdy = (e_n - e_s) / dy
                dzdx = (e_e - e_w) / dx
                slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))
                slope[~np.isfinite(slope)] = np.nan
                domain_features["slope_deg"] = slope
                elev_log.append({"source": "srtm", "status": "created_slope", "non_null": int(np.isfinite(slope).sum())})
    except Exception as e:
        elev_log.append({"source": "srtm", "status": "failed", "error": str(e)})
else:
    elev_log.append({"source": "srtm", "status": "disabled"})

pd.DataFrame(elev_log).to_csv(MODEL_DIR / "18_v3_elevation_feature_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(elev_log)


Getting SRTM elevation for 5491 grid centroids...
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802
4 2884802


,source,status,non_null
0,srtm,created_elevation,5485
1,srtm,created_slope,5476


## 8. Optional Open-Meteo fallback, disabled by default

Only use this if SRTM failed and you explicitly set `USE_OPEN_METEO_FALLBACK=True` in the first cell. It uses small chunks, sleeps, caching, and stops after rate-limit errors.


In [10]:
openmeteo_log = []
cache_path = CACHE_DIR / "18_v3_openmeteo_elevation_cache.csv"

if USE_OPEN_METEO_FALLBACK and ("elevation_m" not in domain_features.columns or domain_features["elevation_m"].isna().all()):
    print("Open-Meteo fallback enabled. This is intentionally slow and rate-limit-safe.")
    cache = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame(columns=["grid_id", "elevation_m"])
    cache_map = dict(zip(cache.get("grid_id", []), cache.get("elevation_m", [])))
    missing = domain_features[~domain_features["grid_id"].isin(cache_map.keys())].copy()
    requests_done = 0
    new_rows = []
    for start in range(0, len(missing), OPEN_METEO_CHUNK_SIZE):
        if requests_done >= OPEN_METEO_MAX_REQUESTS:
            openmeteo_log.append({"status": "stopped_max_requests", "requests_done": requests_done})
            break
        chunk = missing.iloc[start:start+OPEN_METEO_CHUNK_SIZE]
        try:
            params = {
                "latitude": ",".join([f"{x:.5f}" for x in chunk["grid_lat"].values]),
                "longitude": ",".join([f"{x:.5f}" for x in chunk["grid_lon"].values]),
            }
            r = requests.get("https://api.open-meteo.com/v1/elevation", params=params, timeout=60)
            if r.status_code == 429:
                openmeteo_log.append({"status": "rate_limited_429_stop", "start": start, "requests_done": requests_done})
                break
            r.raise_for_status()
            js = r.json()
            vals = js.get("elevation", [])
            if len(vals) == len(chunk):
                for gid, val in zip(chunk["grid_id"].values, vals):
                    new_rows.append({"grid_id": gid, "elevation_m": val})
            requests_done += 1
            time.sleep(OPEN_METEO_SLEEP_SEC)
        except Exception as e:
            openmeteo_log.append({"status": "failed_stop", "start": start, "error": str(e), "requests_done": requests_done})
            break
    if new_rows:
        cache2 = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates("grid_id", keep="last")
        cache2.to_csv(cache_path, index=False, encoding="utf-8-sig")
        domain_features = domain_features.merge(cache2, on="grid_id", how="left", suffixes=("", "_openmeteo"))
        if "elevation_m_openmeteo" in domain_features.columns:
            domain_features["elevation_m"] = domain_features.get("elevation_m", np.nan)
            domain_features["elevation_m"] = domain_features["elevation_m"].fillna(domain_features["elevation_m_openmeteo"])
            domain_features = domain_features.drop(columns=["elevation_m_openmeteo"])
        openmeteo_log.append({"status": "created_or_updated_cache", "new_rows": len(new_rows)})
else:
    openmeteo_log.append({"status": "disabled_or_not_needed"})

pd.DataFrame(openmeteo_log).to_csv(MODEL_DIR / "18_v3_openmeteo_fallback_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(openmeteo_log)


,status
0,disabled_or_not_needed


## 9. Add placeholder columns for target features not yet supported by automatic public data

In [11]:
# Add missing target columns as NaN to make it explicit what remains unresolved.
target_cols = [
    "dist_to_waterbody_km",
    "dist_to_river_km",
    "dist_to_wetland_km",
    "dist_to_coast_km",
    "paddy_ratio_5km",
    "farmland_ratio_5km",
    "forest_ratio_5km",
    "urban_ratio_5km",
    "waterbody_ratio_5km",
    "elevation_m",
    "slope_deg",
    "poultry_farm_density_10km",
    "poultry_farms_in_grid",
    "dist_to_migratory_bird_site_km",
    "dist_to_wildbird_surveillance_site_km",
    "wildbird_site_count_30km",
]
for c in target_cols:
    if c not in domain_features.columns:
        domain_features[c] = np.nan

# For now, Natural Earth cannot distinguish paddy/farmland/forest. Leave as NaN until Japanese land-use data is added.
# If urban_ratio_5km exists from Natural Earth, keep it.

# If waterbody distance exists, ensure lake distance exists.
if "dist_to_lake_or_reservoir_km" not in domain_features.columns:
    domain_features["dist_to_lake_or_reservoir_km"] = domain_features["dist_to_waterbody_km"]


## 10. Validate and save outputs for notebook 14

In [12]:
# Remove forbidden columns
forbidden_cols = [c for c in domain_features.columns if c not in ["grid_id"] and is_forbidden_col(c)]
if forbidden_cols:
    domain_features = domain_features.drop(columns=forbidden_cols)

# Keep grid columns + numeric columns
keep_cols = ["grid_id", "grid_lat", "grid_lon"]
numeric_cols = []
for c in domain_features.columns:
    if c in keep_cols:
        continue
    if pd.api.types.is_numeric_dtype(domain_features[c]):
        numeric_cols.append(c)

domain_features = domain_features[keep_cols + numeric_cols].drop_duplicates("grid_id")
feature_cols = [c for c in domain_features.columns if c not in keep_cols]

validation = pd.DataFrame({
    "metric": ["n_rows", "n_unique_grid_id", "n_feature_cols_excluding_grid_lat_lon", "n_forbidden_cols_removed"],
    "value": [len(domain_features), domain_features["grid_id"].nunique(), len(feature_cols), len(forbidden_cols)]
})
feature_report = pd.DataFrame([{
    "column": c,
    "created": bool(domain_features[c].notna().any()),
    "non_null": int(domain_features[c].notna().sum()),
    "missing_rate": float(domain_features[c].isna().mean()),
    "min": float(pd.to_numeric(domain_features[c], errors="coerce").min()) if pd.to_numeric(domain_features[c], errors="coerce").notna().any() else np.nan,
    "max": float(pd.to_numeric(domain_features[c], errors="coerce").max()) if pd.to_numeric(domain_features[c], errors="coerce").notna().any() else np.nan,
    "mean": float(pd.to_numeric(domain_features[c], errors="coerce").mean()) if pd.to_numeric(domain_features[c], errors="coerce").notna().any() else np.nan,
} for c in feature_cols])

created_list = feature_report[["column", "created", "non_null", "missing_rate"]].copy()

validation.to_csv(MODEL_DIR / "18_v3_domain_feature_validation_report.csv", index=False, encoding="utf-8-sig")
feature_report.to_csv(MODEL_DIR / "18_v3_domain_feature_column_report.csv", index=False, encoding="utf-8-sig")
created_list.to_csv(MODEL_DIR / "18_v3_created_domain_feature_list.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"forbidden_columns_removed": forbidden_cols}).to_csv(MODEL_DIR / "18_v3_forbidden_columns_detected.csv", index=False, encoding="utf-8-sig")

main_csv = DOMAIN_DIR / "grid_environment_features.csv"
main_parquet = DOMAIN_DIR / "grid_environment_features.parquet"
check_csv = MODEL_DIR / "18_v3_grid_environment_features_for_14.csv"
check_parquet = MODEL_DIR / "18_v3_grid_environment_features_for_14.parquet"

domain_features.to_csv(main_csv, index=False, encoding="utf-8-sig")
domain_features.to_parquet(main_parquet, index=False)
domain_features.to_csv(check_csv, index=False, encoding="utf-8-sig")
domain_features.to_parquet(check_parquet, index=False)

print("Created feature columns with non-null values:")
display(created_list[created_list["created"] == True])
print("Saved:", main_csv)
validation


Created feature columns with non-null values:


,column,created,non_null,missing_rate
0,dist_to_coast_km,True,5491,0.000000
1,dist_to_river_km,True,5491,0.000000
2,dist_to_waterbody_km,True,5491,0.000000
3,dist_to_lake_or_reservoir_km,True,5491,0.000000
4,waterbody_ratio_5km,True,5491,0.000000
5,urban_ratio_5km,True,5491,0.000000
6,elevation_m,True,5485,0.001093
7,slope_deg,True,5476,0.002732


Saved: /content/drive/MyDrive/avian_influenza_project/processed/domain_features/grid_environment_features.csv


,metric,value
0,n_rows,5491
1,n_unique_grid_id,5491
2,n_feature_cols_excluding_grid_lat_lon,17
3,n_forbidden_cols_removed,0


## 11. Summary report and saved-file check

In [13]:
created_nonnull = created_list[created_list["created"] == True]["column"].tolist()
not_created = created_list[created_list["created"] == False]["column"].tolist()

report = []
report.append("# 18 v3 Summary: rate-limit-safe automatic public GIS/domain feature construction")
report.append("")
report.append(f"Generated by `{NOTEBOOK_VERSION}`.")
report.append("")
report.append("## Created features")
report.append(f"- Number of created feature columns excluding `grid_id`, `grid_lat`, `grid_lon`: {len(created_nonnull)}")
for c in created_nonnull:
    non_null = int(created_list.loc[created_list.column == c, "non_null"].iloc[0])
    report.append(f"- `{c}`: non-null {non_null}")
report.append("")
report.append("## Not yet created")
for c in not_created:
    report.append(f"- `{c}`")
report.append("")
report.append("## Notes")
report.append("- Natural Earth-derived features are coarse global GIS features. For publication-quality Japan-specific analysis, replace or supplement them with 国土数値情報 and GSI data.")
report.append("- Open-Meteo Elevation fallback is disabled by default to avoid 429 Too Many Requests errors.")
report.append("- If SRTM elevation failed, place a DEM GeoTIFF under `processed/environment_raw` or rerun with a local DEM pipeline.")
report.append("- After this notebook, rerun notebook 14 to evaluate whether these domain features improve strict first-occurrence risk mapping.")

summary_path = MODEL_DIR / "18_v3_summary_report_auto_public_gis_domain_feature_construction.md"
summary_path.write_text("\n".join(report), encoding="utf-8")

expected = [
    DOMAIN_DIR / "grid_environment_features.csv",
    DOMAIN_DIR / "grid_environment_features.parquet",
    MODEL_DIR / "18_v3_grid_environment_features_for_14.csv",
    MODEL_DIR / "18_v3_created_domain_feature_list.csv",
    MODEL_DIR / "18_v3_domain_feature_column_report.csv",
    MODEL_DIR / "18_v3_summary_report_auto_public_gis_domain_feature_construction.md",
]
saved = pd.DataFrame([{
    "path": str(p),
    "exists": p.exists(),
    "size_bytes": p.stat().st_size if p.exists() else 0
} for p in expected])
saved.to_csv(MODEL_DIR / "18_v3_saved_file_check.csv", index=False, encoding="utf-8-sig")

print(summary_path)
display(saved)
print("Done. Rerun notebook 14 next.")


/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/18_v3_summary_report_auto_public_gis_domain_feature_construction.md


,path,exists,size_bytes
0,/content/drive/MyDrive/avian_influenza_project...,True,776946
1,/content/drive/MyDrive/avian_influenza_project...,True,327812
2,/content/drive/MyDrive/avian_influenza_project...,True,776946
3,/content/drive/MyDrive/avian_influenza_project...,True,650
4,/content/drive/MyDrive/avian_influenza_project...,True,1011
5,/content/drive/MyDrive/avian_influenza_project...,True,1361


Done. Rerun notebook 14 next.


In [14]:
pd.read_csv(
    "/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/18_v3_created_domain_feature_list.csv"
)

,column,created,non_null,missing_rate
0,dist_to_coast_km,True,5491,0.000000
1,dist_to_river_km,True,5491,0.000000
2,dist_to_waterbody_km,True,5491,0.000000
3,dist_to_lake_or_reservoir_km,True,5491,0.000000
4,waterbody_ratio_5km,True,5491,0.000000
5,urban_ratio_5km,True,5491,0.000000
6,elevation_m,True,5485,0.001093
7,slope_deg,True,5476,0.002732
8,dist_to_wetland_km,False,0,1.000000
9,paddy_ratio_5km,False,0,1.000000
